In [1]:
import numpy as np
from pathlib import Path
import sys
import os
import xarray as xr

In [10]:
root_dir = Path(os.getcwd()).parent.parent.parent
data_dir = root_dir / 'data' / 'ca_imaging'

# Add the model directory to Python path
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.data_pipeline.preprocessing.preprocess import normalize99

Load original data

In [3]:
filenames = [data_dir / "cells_train.npz",
              data_dir / "cells_test.npz"]

cells_train = np.load(data_dir / 'cells_train.npz', allow_pickle=True)['arr_0'].item()
cells_test = np.load(data_dir / 'cells_test.npz', allow_pickle=True)['arr_0'].item()

images_train = np.array(cells_train['imgs']).transpose(0, 3, 1, 2)
masks_train = np.array(cells_train['masks'])
images_test = np.array(cells_test['imgs']).transpose(0, 3, 1, 2)
masks_test = np.array(cells_test['masks'])

Concatenate training and test data

In [4]:
images = np.concatenate([images_train, images_test], axis = 0)
masks = np.concatenate([masks_train, masks_test], axis = 0)
# make cell present/no cell present labels for each pixel
labels = (masks > 0).astype(np.longlong)

Transform images and labels to square shapes

In [7]:
import cv2
import numpy as np

# Your image dimensions
nimages, nchan, height, width = images.shape
new_dims = (448, 448)

# Define source points (corners of center crop in original image)
crop_h, crop_w = new_dims[0], new_dims[1]
top = (height - crop_h) // 2
left = (width - crop_w) // 2

# Source points: top-left, top-right, bottom-left of crop region
pts1 = np.float32([
    [left, top],           # top-left
    [left + crop_w, top],  # top-right
    [left, top + crop_h]   # bottom-left
])

# Destination points: map to full output image
pts2 = np.float32([
    [0, 0],           # top-left
    [crop_w, 0],      # top-right
    [0, crop_h]       # bottom-left
])

# Get affine transformation matrix
M = cv2.getAffineTransform(pts1, pts2)

# Apply transformation
images_transformed = np.zeros((nimages, nchan, new_dims[0], new_dims[1]), dtype=images.dtype)
labels_transformed = np.zeros((nimages, new_dims[0], new_dims[1]), dtype=labels.dtype)

for n in range(nimages):
    for k in range(nchan):
        I = cv2.warpAffine(images[n, k], M, (new_dims[1], new_dims[0]), flags=cv2.INTER_LINEAR)
        images_transformed[n, k] = I


    labels_transformed[n] = cv2.warpAffine(labels[n], M, (new_dims[1], new_dims[0]),
                                        flags=cv2.INTER_NEAREST)

Normalize image data

In [11]:
images_transformed = np.array([normalize99(img) for img in images_transformed])

Create xarray DataArrays for images and labels and put them together in xarray Dataset.

In [12]:
images_xarr = xr.DataArray(
        data=images_transformed,
        dims=['image_nr', 'staining', 'height', 'width'],
        coords={'image_nr': ('image_nr', np.arange(images_transformed.shape[0])),
                'staining' : ('staining', ['cytoplasm', 'nuclear'])},
        name='ca_images',
        attrs={
            'description': 'Calcium imaging data with two kinds of staining',
            'notes': 'Cytoplasm means whole cell stained, nuclear means only nucleus of cells stained.'
        }
    )

labels_xarr = xr.DataArray(
        data=labels_transformed,
        dims=['image_nr', 'height', 'width'],
        coords={'image_nr': ('image_nr', np.arange(images_transformed.shape[0]))},
        name='cell_labels',
        attrs={
            'description': 'Cell labels',
            'notes': '1 means there is a cell at pixel, 0 means there is no cell at pixel.'
        }
    )

ds = xr.Dataset({
    'ca_image_data': images_xarr,
    'cell_labels': labels_xarr
})
ds

<xarray.Dataset> Size: 292MB
Dimensions:        (image_nr: 91, staining: 2, height: 448, width: 448)
Coordinates:
  * image_nr       (image_nr) int64 728B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89 90
  * staining       (staining) <U9 72B 'cytoplasm' 'nuclear'
Dimensions without coordinates: height, width
Data variables:
    ca_image_data  (image_nr, staining, height, width) float32 146MB 0.0 ... 0.0
    cell_labels    (image_nr, height, width) int64 146MB 0 0 0 0 0 ... 0 0 0 0 0

Write dataset to file

In [13]:
ds.to_netcdf(data_dir / 'cells_data_all.nc')

Check saved file

In [14]:
dataset_check = xr.load_dataset(data_dir / 'cells_data_all.nc')

dataset_check

<xarray.Dataset> Size: 292MB
Dimensions:        (image_nr: 91, staining: 2, height: 448, width: 448)
Coordinates:
  * image_nr       (image_nr) int64 728B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89 90
  * staining       (staining) <U9 72B 'cytoplasm' 'nuclear'
Dimensions without coordinates: height, width
Data variables:
    ca_image_data  (image_nr, staining, height, width) float32 146MB 0.0 ... 0.0
    cell_labels    (image_nr, height, width) int64 146MB 0 0 0 0 0 ... 0 0 0 0 0